# 🧪 Face Recognition Testing (YOLOv8 + LBPH)
ทดสอบระบบจดจำใบหน้า:
1. โหลดโมเดล LBPH + Config อัตโนมัติจาก `model_config.json`
2. ทดสอบภาพเดี่ยวจาก `test/`
3. ทดสอบ Batch ทั้งโฟลเดอร์ `test/`
4. ทดสอบ Live Webcam (Colab + Local)

In [ ]:
# 1. เชื่อมต่อ Google Drive
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ เชื่อมต่อ Google Drive สำเร็จ")
    IS_COLAB = True
except ImportError:
    print("ℹ️ รันใน Local Environment")
    IS_COLAB = False

In [ ]:
# 2. ติดตั้งและนำเข้าไลบรารี
import subprocess
import sys

def install_if_missing(package, import_name=None):
    try:
        __import__(import_name or package)
    except ImportError:
        print(f"กำลังติดตั้ง {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install_if_missing("opencv-contrib-python", "cv2")
install_if_missing("ultralytics")

import os
import json
import cv2
import cv2.face
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

print(f"✅ OpenCV Version: {cv2.__version__}")

In [ ]:
# 3. ฟังก์ชัน Preprocessing + Confidence
CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocess_face(face_gray, target_size=(120, 120)):
    face_resized = cv2.resize(face_gray, target_size, interpolation=cv2.INTER_CUBIC)
    face_clahe = CLAHE.apply(face_resized)
    face_denoised = cv2.GaussianBlur(face_clahe, (3, 3), 0)
    return face_denoised

def calculate_confidence_pct(distance, threshold):
    """คำนวณ Confidence % แบบ intuitive"""
    if distance <= 0:
        return 100.0
    if distance >= threshold:
        return 0.0
    return max(0.0, (1.0 - distance / threshold) * 100.0)

print("✅ ฟังก์ชัน Preprocessing พร้อมใช้งาน")

In [ ]:
# 4. โหลดโมเดล LBPH + Config + YOLOv8

# --- ค้นหา Config ---
possible_config_paths = [
    'model_config.json', './model_config.json', '../model_config.json',
    '/content/drive/MyDrive/Project_Ai/model_config.json',
    '/content/drive/My Drive/Project_Ai/model_config.json',
    os.path.join(os.getcwd(), 'model_config.json')
]

CONFIG_PATH = None
for p in possible_config_paths:
    if os.path.exists(p):
        CONFIG_PATH = p
        break

if CONFIG_PATH is not None:
    with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
        config = json.load(f)
    label_dict = {int(k): v for k, v in config['label_dict'].items()}
    CONFIDENCE_THRESHOLD = config['optimal_threshold']
    MODEL_FILE = config.get('model_file', 'Project_Ai_model_v2.yml')
    TARGET_SIZE = tuple(config.get('target_size', [120, 120]))
    print(f"✅ โหลด Config สำเร็จจาก: {CONFIG_PATH}")
    print(f"   Threshold: {CONFIDENCE_THRESHOLD}")
    print(f"   F1-Score ตอน Train: {config.get('best_f1_score', 'N/A')}")
    print(f"   Accuracy ตอน Train: {config.get('best_accuracy', 'N/A')}")
else:
    print("⚠️ ไม่พบ model_config.json ใช้ค่า default")
    label_dict = {0: "Captun", 1: "Dream", 2: "Gun", 3: "Max", 4: "Rung"}
    CONFIDENCE_THRESHOLD = 55.0
    MODEL_FILE = 'Project_Ai_model_v2.yml'
    TARGET_SIZE = (120, 120)

print(f"📋 รายชื่อ: {label_dict}")

# --- โหลดโมเดล LBPH ---
config_dir = os.path.dirname(CONFIG_PATH) if CONFIG_PATH else '.'
possible_model_paths = [
    os.path.join(config_dir, MODEL_FILE),
    MODEL_FILE, f'./{MODEL_FILE}', f'../{MODEL_FILE}',
    f'/content/drive/MyDrive/Project_Ai/{MODEL_FILE}',
    f'/content/drive/My Drive/Project_Ai/{MODEL_FILE}',
    os.path.join(os.getcwd(), MODEL_FILE)
]

MODEL_PATH = None
for p in possible_model_paths:
    if os.path.exists(p):
        MODEL_PATH = p
        break

recognizer = cv2.face.LBPHFaceRecognizer_create()
if MODEL_PATH is not None:
    recognizer.read(MODEL_PATH)
    print(f"✅ โหลดโมเดล LBPH สำเร็จจาก: {MODEL_PATH}")
else:
    raise FileNotFoundError(f"❌ ไม่พบไฟล์โมเดล {MODEL_FILE}")

# --- โหลด YOLOv8 ---
print("กำลังโหลดโมเดล YOLOv8...")
yolo_model = YOLO('yolov8n.pt')
print("✅ โหลดโมเดล YOLOv8 สำเร็จ!")

# --- โหลด Haar Cascades ---
cascade_paths = [
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml',
    cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml',
    cv2.data.haarcascades + 'haarcascade_profileface.xml',
]
cascades = []
for cp in cascade_paths:
    c = cv2.CascadeClassifier(cp)
    if not c.empty():
        cascades.append(c)

face_cascade = cascades[0] if cascades else cv2.CascadeClassifier()

def detect_faces_multi(gray_img, min_size=(30, 30)):
    """ตรวจจับใบหน้าด้วยหลาย cascade"""
    all_faces = []
    for cascade in cascades:
        faces = cascade.detectMultiScale(gray_img, scaleFactor=1.1, minNeighbors=4, minSize=min_size)
        if len(faces) > 0:
            all_faces.extend(faces.tolist())
    if len(all_faces) == 0:
        for cascade in cascades:
            faces = cascade.detectMultiScale(gray_img, scaleFactor=1.05, minNeighbors=3, minSize=(20,20))
            if len(faces) > 0:
                all_faces.extend(faces.tolist())
                break
    # Simple NMS
    if len(all_faces) <= 1:
        return all_faces
    unique = []
    used = set()
    for i, (x1,y1,w1,h1) in enumerate(all_faces):
        if i in used: continue
        best = (x1,y1,w1,h1)
        best_a = w1*h1
        for j, (x2,y2,w2,h2) in enumerate(all_faces):
            if j <= i or j in used: continue
            ox = max(0, min(x1+w1,x2+w2)-max(x1,x2))
            oy = max(0, min(y1+h1,y2+h2)-max(y1,y2))
            overlap = ox*oy
            ma = min(w1*h1, w2*h2)
            if ma > 0 and overlap/ma > 0.5:
                used.add(j)
                if w2*h2 > best_a:
                    best = (x2,y2,w2,h2)
                    best_a = w2*h2
        unique.append(best)
    return unique

print("✅ ระบบพร้อมใช้งาน")

### 👤 5. ฟังก์ชันระบุตัวตน YOLOv8 + LBPH (พร้อม Fallback)

In [ ]:
# 5. ฟังก์ชันหลักระบุตัวตน
def recognize_and_display(image_input, confidence_threshold=CONFIDENCE_THRESHOLD,
                          title_prefix="Face Recognition"):
    if isinstance(image_input, str):
        if not os.path.exists(image_input):
            print(f"❌ ไม่พบไฟล์ภาพ: {image_input}")
            return
        img = cv2.imread(image_input)
    else:
        img = image_input.copy()

    if img is None:
        print("❌ ไม่สามารถเปิดภาพได้")
        return

    display_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h_img, w_img = img.shape[:2]

    # 1. YOLOv8 ตรวจจับบุคคล
    results = yolo_model(img, verbose=False, classes=[0])
    detected_faces = 0

    for r in results:
        for box in r.boxes:
            bx1, by1, bx2, by2 = map(int, box.xyxy[0])
            bx1, by1 = max(0, bx1), max(0, by1)
            bx2, by2 = min(w_img, bx2), min(h_img, by2)

            person_roi = gray[by1:by2, bx1:bx2]
            if person_roi.size == 0: continue

            faces = detect_faces_multi(person_roi)
            for (fx, fy, fw, fh) in faces:
                pad_x, pad_y = int(0.05*fw), int(0.05*fh)
                ax1 = max(0, bx1+fx-pad_x)
                ay1 = max(0, by1+fy-pad_y)
                ax2 = min(w_img, bx1+fx+fw+pad_x)
                ay2 = min(h_img, by1+fy+fh+pad_y)

                face_crop = gray[ay1:ay2, ax1:ax2]
                if face_crop.size == 0: continue

                face_proc = preprocess_face(face_crop, TARGET_SIZE)
                pred_id, dist = recognizer.predict(face_proc)
                conf_pct = calculate_confidence_pct(dist, confidence_threshold)

                if dist <= confidence_threshold:
                    name = label_dict.get(pred_id, f"ID:{pred_id}")
                    color = (0, 255, 0)
                    lbl = f"{name} {conf_pct:.0f}% (D:{dist:.1f})"
                else:
                    name = "Unknown"
                    color = (255, 0, 0)
                    lbl = f"{name} (D:{dist:.1f})"

                cv2.rectangle(display_img, (ax1,ay1), (ax2,ay2), color, 3)
                bw = int(len(lbl)*12)
                cv2.rectangle(display_img, (ax1, max(0,ay1-25)), (ax1+bw, max(0,ay1)), color, -1)
                cv2.putText(display_img, lbl, (ax1+5, max(15,ay1-7)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
                detected_faces += 1

    # Fallback ถ้า YOLO ไม่พบ
    if detected_faces == 0:
        faces_fb = detect_faces_multi(gray)
        for (x, y, w, h) in faces_fb:
            pad_x, pad_y = int(0.05*w), int(0.05*h)
            x1, y1 = max(0, x-pad_x), max(0, y-pad_y)
            x2, y2 = min(w_img, x+w+pad_x), min(h_img, y+h+pad_y)
            face_crop = gray[y1:y2, x1:x2]
            if face_crop.size == 0: continue

            face_proc = preprocess_face(face_crop, TARGET_SIZE)
            pred_id, dist = recognizer.predict(face_proc)
            conf_pct = calculate_confidence_pct(dist, confidence_threshold)

            if dist <= confidence_threshold:
                name = label_dict.get(pred_id, "Unknown")
                color = (0, 255, 0)
                lbl = f"{name} {conf_pct:.0f}% (D:{dist:.1f})"
            else:
                name = "Unknown"
                color = (255, 0, 0)
                lbl = f"{name} (D:{dist:.1f})"

            cv2.rectangle(display_img, (x1,y1), (x2,y2), color, 3)
            bw = int(len(lbl)*12)
            cv2.rectangle(display_img, (x1, max(0,y1-25)), (x1+bw, max(0,y1)), color, -1)
            cv2.putText(display_img, lbl, (x1+5, max(15,y1-7)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
            detected_faces += 1

    print(f"🔍 ตรวจพบ {detected_faces} ใบหน้า")
    plt.figure(figsize=(10, 8))
    plt.imshow(display_img)
    plt.title(f"{title_prefix} (Threshold: {confidence_threshold:.1f})")
    plt.axis('off')
    plt.show()

# ทดสอบภาพแรกจากโฟลเดอร์ test
test_folders = ['test', './test', '../test',
                '/content/drive/MyDrive/Project_Ai/test',
                '/content/drive/My Drive/Project_Ai/test']
for tf in test_folders:
    if os.path.exists(tf) and len(os.listdir(tf)) > 0:
        sample = sorted(os.listdir(tf))[0]
        recognize_and_display(os.path.join(tf, sample), title_prefix="Single Image Test")
        break

### 📂 6. Batch Testing ทุกภาพในโฟลเดอร์ test/

In [ ]:
# 6. Batch Testing (พร้อม Fallback)
possible_test_dirs = ['test', './test', '../test',
                      '/content/drive/MyDrive/Project_Ai/test',
                      '/content/drive/My Drive/Project_Ai/test']
TEST_DIR = None
for d in possible_test_dirs:
    if os.path.exists(d) and len(os.listdir(d)) > 0:
        TEST_DIR = d
        break

if TEST_DIR is None:
    print("⚠️ ไม่พบโฟลเดอร์ test")
else:
    test_images = sorted([f for f in os.listdir(TEST_DIR)
                         if f.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f"📂 พบภาพทดสอบ {len(test_images)} ภาพ ใน '{TEST_DIR}'")

    if len(test_images) > 0:
        cols = 3
        rows = (len(test_images) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(16, 5*rows))
        if rows * cols == 1:
            axes = np.array([axes])
        axes = np.array(axes).flatten()

        for idx, img_name in enumerate(test_images):
            ax = axes[idx]
            img_path = os.path.join(TEST_DIR, img_name)
            img = cv2.imread(img_path)
            if img is None: continue

            display_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            h_img, w_img = img.shape[:2]

            results = yolo_model(img, verbose=False, classes=[0])
            status_texts = []
            detected = 0

            for r in results:
                for box in r.boxes:
                    bx1,by1,bx2,by2 = map(int, box.xyxy[0])
                    bx1,by1 = max(0,bx1), max(0,by1)
                    bx2,by2 = min(w_img,bx2), min(h_img,by2)

                    person_roi = gray[by1:by2, bx1:bx2]
                    if person_roi.size == 0: continue

                    faces = detect_faces_multi(person_roi)
                    for (fx,fy,fw,fh) in faces:
                        pad_x,pad_y = int(0.05*fw), int(0.05*fh)
                        ax1 = max(0, bx1+fx-pad_x)
                        ay1 = max(0, by1+fy-pad_y)
                        ax2 = min(w_img, bx1+fx+fw+pad_x)
                        ay2 = min(h_img, by1+fy+fh+pad_y)

                        face_crop = gray[ay1:ay2, ax1:ax2]
                        if face_crop.size == 0: continue

                        face_proc = preprocess_face(face_crop, TARGET_SIZE)
                        pred_id, dist = recognizer.predict(face_proc)
                        conf_pct = calculate_confidence_pct(dist, CONFIDENCE_THRESHOLD)

                        if dist <= CONFIDENCE_THRESHOLD:
                            name = label_dict.get(pred_id, "Unknown")
                            color = (0,255,0)
                            lbl_text = f"{name} {conf_pct:.0f}%"
                        else:
                            name = "Unknown"
                            color = (255,0,0)
                            lbl_text = f"Unknown ({dist:.0f})"

                        status_texts.append(f"{name} ({dist:.1f})")
                        cv2.rectangle(display_img, (ax1,ay1), (ax2,ay2), color, 3)
                        bw = int(len(lbl_text)*12)
                        cv2.rectangle(display_img, (ax1,max(0,ay1-22)), (ax1+bw,max(0,ay1)), color, -1)
                        cv2.putText(display_img, lbl_text, (ax1+4,max(14,ay1-6)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255,255,255), 2)
                        detected += 1

            # Fallback ถ้า YOLO ไม่พบ
            if detected == 0:
                faces_fb = detect_faces_multi(gray)
                for (x,y,w,h) in faces_fb:
                    pad_x,pad_y = int(0.05*w), int(0.05*h)
                    x1,y1 = max(0,x-pad_x), max(0,y-pad_y)
                    x2,y2 = min(w_img,x+w+pad_x), min(h_img,y+h+pad_y)
                    face_crop = gray[y1:y2, x1:x2]
                    if face_crop.size == 0: continue
                    face_proc = preprocess_face(face_crop, TARGET_SIZE)
                    pred_id, dist = recognizer.predict(face_proc)
                    conf_pct = calculate_confidence_pct(dist, CONFIDENCE_THRESHOLD)
                    if dist <= CONFIDENCE_THRESHOLD:
                        name = label_dict.get(pred_id, "Unknown")
                        color = (0,255,0)
                        lbl_text = f"{name} {conf_pct:.0f}%"
                    else:
                        name = "Unknown"
                        color = (255,0,0)
                        lbl_text = f"Unknown ({dist:.0f})"
                    status_texts.append(f"{name} ({dist:.1f})")
                    cv2.rectangle(display_img, (x1,y1), (x2,y2), color, 3)
                    bw = int(len(lbl_text)*12)
                    cv2.rectangle(display_img, (x1,max(0,y1-22)), (x1+bw,max(0,y1)), color, -1)
                    cv2.putText(display_img, lbl_text, (x1+4,max(14,y1-6)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255,255,255), 2)

            ax.imshow(display_img)
            title = f"{img_name}\n" + (", ".join(status_texts) if status_texts else "No Face")
            ax.set_title(title, fontsize=9)
            ax.axis('off')

        for k in range(len(test_images), len(axes)):
            axes[k].axis('off')

        plt.tight_layout()
        plt.show()
        print("✅ Batch Testing เสร็จสิ้น!")

### 📸 7. Live Webcam (Google Colab)

In [ ]:
# 7. เปิดกล้องบน Google Colab
from IPython.display import display, Javascript
from base64 import b64decode

def capture_photo_colab(quality=0.85):
    js = Javascript("""
        async function takePhoto(quality) {
            const div = document.createElement('div');
            div.style.padding = '10px';
            div.style.backgroundColor = '#f0f4f8';
            div.style.borderRadius = '10px';
            div.style.display = 'inline-block';
            div.style.textAlign = 'center';

            const title = document.createElement('h3');
            title.textContent = '📷 ถ่ายภาพเพื่อระบุตัวตน';
            div.appendChild(title);

            const video = document.createElement('video');
            video.style.display = 'block';
            video.style.borderRadius = '8px';
            video.style.maxWidth = '480px';
            div.appendChild(video);

            const capture = document.createElement('button');
            capture.textContent = '📸 ถ่ายภาพ';
            capture.style.padding = '12px 25px';
            capture.style.fontSize = '16px';
            capture.style.marginTop = '15px';
            capture.style.cursor = 'pointer';
            div.appendChild(capture);

            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(div);
            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    """)
    display(js)
    try:
        from google.colab.output import eval_js
        data = eval_js('takePhoto({})'.format(quality))
        binary = b64decode(data.split(',')[1])
        img_arr = np.frombuffer(binary, dtype=np.uint8)
        img_cv = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
        return img_cv
    except Exception as e:
        print(f"⚠️ ไม่สามารถเปิดกล้องบน Colab: {e}")
        return None

try:
    print("กำลังเปิดกล้อง... กรุณากดยอมรับ (Allow)")
    captured = capture_photo_colab()
    if captured is not None:
        print("✅ ถ่ายภาพสำเร็จ!")
        recognize_and_display(captured, title_prefix="Live Webcam")
except Exception as e:
    print(f"ℹ️ {e}")

### 💻 8. Live Webcam (Local PC)

In [ ]:
# 8. เปิดกล้องบนเครื่อง Local
def test_local_webcam():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ ไม่สามารถเปิดกล้องได้")
        return

    print("🎥 เปิดกล้องสำเร็จ:")
    print("  กด [SPACE] ถ่ายภาพ + ระบุตัวตน")
    print("  กด [Q] ปิดกล้อง")

    while True:
        ret, frame = cap.read()
        if not ret: break

        cv2.putText(frame, "SPACE=Capture | Q=Quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
        cv2.imshow("Live Webcam", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord(' '):
            captured_frame = frame.copy()  # คัดลอกก่อน release
            cap.release()
            cv2.destroyAllWindows()
            print("📸 ถ่ายภาพแล้ว กำลังประมวลผล...")
            recognize_and_display(captured_frame, title_prefix="Local Webcam")
            break
        elif key == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            print("ปิดกล้องเรียบร้อย")
            break

# ปลดคอมเมนต์บรรทัดล่างเมื่อต้องการทดสอบ:
# test_local_webcam()